# Phase 5: k-NN Clean Recommendation Engine & Guardrails (v3.1 - Fixed Energy Drink vs Energy Bar Keyword Collision)

Maps each of the 126 scanned Indian packaged products (Phase 4) to clean Indian D2C
startup alternatives using a **Strict Human-Centric Food-Type Guardrail System**.

### Bugfix v3.1:
- Fixed `SODA_KEYWORDS` collision: Changed `'energy'` to `'energy drink'` so that **Energy Bars** (e.g. *Yoga Bar Multigrain Energy Bar*) get scored as **Snack Bars (Base 85, Score 85)** instead of being misclassified as **Energy Drinks (Base 70)**.


In [1]:
import pandas as pd, json, re, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

with open('data/health_scores.json', 'r', encoding='utf-8') as f:
    scanned_products = json.load(f)
df_scanned = pd.DataFrame(scanned_products)

with open('data/clean_indian_startup_brands.json', 'r', encoding='utf-8') as f:
    seed_brands = json.load(f)
df_seed = pd.DataFrame([s for s in seed_brands if 'brand' in s])

print(f"Loaded {len(df_scanned)} scanned products (Phase 4).")
print(f"Loaded {len(df_seed)} clean Indian D2C seed products.")


Loaded 126 scanned products (Phase 4).
Loaded 51 clean Indian D2C seed products.


In [2]:
def calculate_thrs_v2_for_seeds(row):
    """Score seed brands by reading ingredients_raw directly — not key_difference."""
    item_name   = str(row.get('item_name',   '')).lower()
    ingredients = str(row.get('ingredients_raw', '')).lower()
    decoded     = row.get('decoded_e_numbers', {})
    if not isinstance(decoded, dict):
        decoded = {}

    # Bugfix v3.1: Use 'energy drink' instead of plain 'energy' to prevent collision with 'energy bar'
    SODA_KEYWORDS = ['drink','soda','coke','pepsi','cola','limca',
                     'sprite','dew','7up','7 up','fanta','mirinda','energy drink']
    if any(re.search(r'\b' + re.escape(k) + r'\b', item_name) for k in SODA_KEYWORDS):
        base_score = 70
    elif any(k in item_name for k in ['chocolate','candy','toffee','biscuit','cookie',
                                        'waffle','cake','sweet','bar','muesli',
                                        'pancake','peanut butter','spread']):
        base_score = 85
    else:
        base_score = 95

    REFINED_SUGARS   = ['glucose syrup','corn syrup','high fructose','dextrose','maltose','maltodextrin','fructose']
    NATURAL_SWEETENERS = ['jaggery','dates','honey','coconut sugar','unrefined sugar',
                          'raw sugar','muscovado','brown rice syrup','stevia','sugarcane juice']
    sugar_count = sum(1 for t in REFINED_SUGARS if t in ingredients)
    if re.search(r'\bsugar\b', ingredients) and not any(n in ingredients for n in NATURAL_SWEETENERS):
        sugar_count += 1
    sugar_penalty = min(15, sugar_count * 5)

    PRESERVATIVES = ['sodium benzoate','potassium sorbate','ins 202','ins 211','(202)','(211)',' 202',' 211']
    preservative_penalty = 15 if any(k in ingredients for k in PRESERVATIVES) else 0

    color_count = sum(1 for e in decoded.values()
        if any(c in str(e).lower() for c in ['tartrazine','carmoisine','azorubine','blue','yellow','red','color']))
    color_penalty = min(15, color_count * 10)

    fat_penalty       = 10 if ('palmolein' in ingredients or 'palm oil' in ingredients or 'hydrogenated' in ingredients) else 0
    emulsifier_penalty= 10 if ('ins 476' in ingredients or 'pgpr' in ingredients or 'ins 442' in ingredients
                                or any('476' in k for k in decoded) or any('442' in k for k in decoded)) else 0

    return max(0, base_score - (sugar_penalty + preservative_penalty + color_penalty + fat_penalty + emulsifier_penalty))

df_seed['thrs_v2_score'] = df_seed.apply(calculate_thrs_v2_for_seeds, axis=1)

print("=" * 65)
print("SEED BRANDS — CORRECTED THRS v2.0 SCORES (v3.1)")
print("=" * 65)
print(df_seed[['brand','item_name','category','thrs_v2_score']].to_string())


SEED BRANDS — CORRECTED THRS v2.0 SCORES (v3.1)
              brand                                             item_name                category  thrs_v2_score
0        Paper Boat                   Paper Boat Aamras Mango Fruit Juice               Beverages             80
1        Paper Boat                             Paper Boat Anardana Juice               Beverages             95
2        Paper Boat                          Paper Boat Jamun Kala Khatta               Beverages             95
3        Paper Boat                              Paper Boat Coconut Water               Beverages             90
4        Paper Boat                          Paper Boat Swing Slurp Mango               Beverages             90
5        Paper Boat                                   Paper Boat Jaljeera               Beverages             95
6        Paper Boat                                 Paper Boat Badam Milk               Beverages             90
7      Raw Pressery                Raw Pressery 

In [3]:
SODA_KEYWORDS = ['drink','soda','coke','pepsi','cola','limca',
                  'sprite','dew','7up','7 up','fanta','mirinda','energy drink','juice','beverage']
SWEET_KEYWORDS = ['chocolate','candy','toffee','biscuit','cookie','waffle',
                   'cake','sweet','bar','muesli','pancake','peanut butter',
                   'bournvita','bites','spread','brownie']

def get_canonical_category(item_name: str) -> str:
    name = item_name.lower()
    if any(re.search(r'\b' + re.escape(k) + r'\b', name) for k in SODA_KEYWORDS):
        return "Beverages / Sodas"
    elif any(k in name for k in SWEET_KEYWORDS):
        return "Confectionery / Sweets"
    else:
        return "General / Grocery"

df_scanned['canonical_category'] = df_scanned['item_name'].apply(get_canonical_category)
print("Scanned product category distribution:")
print(df_scanned['canonical_category'].value_counts())


Scanned product category distribution:
canonical_category
Confectionery / Sweets    66
General / Grocery         42
Beverages / Sodas         18
Name: count, dtype: int64


In [4]:
FOOD_TYPE_RULES = [
    ('noodles',          ['noodle', 'maggi', 'pasta', 'ramen', 'cuppa noodles', 'chow']),
    ('juice_beverage',   ['juice', 'squash', 'aam panna', 'tropicana', 'real', 'paper boat',
                          'raw pressery', 'drink', 'beverage', 'nectar', 'slush']),
    ('soda_carbonated',  ['\bcola\b', '\bdew\b', '\bpepsi\b', '\bcoke\b', '\bsprite\b',
                          '\blimca\b', '\bsoda\b', '\bfanta\b', '7up', '7 up', 'mirinda']),
    ('chocolate_bar',    ['chocolate', 'choco', 'cocoa', 'bournville', 'dairy milk', 'kitkat',
                          'snickers', 'bounty', 'toblerone', 'ferrero', 'lindt', 'temptations',
                          '5 star', 'munch', 'perk', 'milkybar', 'hershey']),
    ('chocolate_spread', ['nutella', 'chocolate spread', 'choco spread', 'cocoa spread', 'hazelnut spread']),
    ('peanut_butter',    ['peanut butter', 'nut butter', 'almond butter']),
    ('protein_bar',      ['protein bar', 'energy bar', 'granola bar', 'cereal bar']),
    ('muesli_cereal',    ['muesli', 'granola', 'oats', 'cereal', 'flakes', 'corn flakes']),
    ('chips_crisps',     ['chips', 'crisps', 'bingo', 'lays', 'doritos', 'kurkure', 'namkeen', 'makhana', 'wafer']),
    ('biscuit_cookie',   ['biscuit', 'cookie', 'digestive', 'hide & seek', 'oreo', 'bourbon', 'parle-g',
                          'good day', 'krackjack', 'monaco', 'nutrichoice', 'crack']),
    ('cake_baking',      ['cake', 'brownie', 'muffin', 'cake mix', 'brownie mix', 'waffle', 'pancake']),
    ('yogurt_dairy',     ['yogurt', 'curd', 'dahi', 'greek yogurt', 'lassi', 'milkshake', 'butter', 'mayo', 'mayonnaise']),
]

SEED_FOOD_TYPES = {
    'peanut butter':          'peanut_butter',
    'protein bar':            'protein_bar',
    'protein concentrate':    'protein_bar',
    'energy bar':             'protein_bar',
    'dark chocolate bar':     'chocolate_bar',
    '87% dark chocolate':     'chocolate_bar',
    'dark chocolate peanut':  'chocolate_bar',
    'muesli':                 'muesli_cereal',
    'banana chips':           'chips_crisps',
    'greek yogurt':           'yogurt_dairy',
    'millet noodles':         'noodles',
    'foxtail millet noodles': 'noodles',
    'pancake mix':            'pancake_mix',
    'cake mix':               'pancake_mix',
    'sugarcane juice':        'juice_beverage',
    'orange juice':           'juice_beverage',
    'coconut water':          'juice_beverage',
}

def detect_food_type(item_name: str) -> str:
    name = item_name.lower()
    for food_type, keywords in FOOD_TYPE_RULES:
        for kw in keywords:
            if kw.startswith('\b'):
                if re.search(kw, name):
                    return food_type
            elif kw in name:
                return food_type
    return 'general'

def get_seed_food_type(item_name: str) -> str:
    name = item_name.lower()
    for fragment, ftype in SEED_FOOD_TYPES.items():
        if fragment in name:
            return ftype
    return 'general'

df_seed['food_type'] = df_seed['item_name'].apply(get_seed_food_type)

def recommend_alternatives(scanned_row, df_seed):
    item_name = str(scanned_row.get('item_name',''))
    canonical_category = get_canonical_category(item_name)
    scanned_food_type  = detect_food_type(item_name)

    type_pool = df_seed[df_seed['food_type'] == scanned_food_type].copy()
    if type_pool.empty:
        FLEXIBLE_MAPPINGS = {
            'soda_carbonated': ['juice_beverage'],
            'cake_baking':     ['pancake_mix', 'chocolate_bar'],
            'biscuit_cookie':  ['pancake_mix', 'chocolate_bar', 'muesli_cereal'],
            'chocolate_spread':['chocolate_bar', 'peanut_butter'],
        }
        rel_types = FLEXIBLE_MAPPINGS.get(scanned_food_type, [])
        type_pool = df_seed[df_seed['food_type'].isin(rel_types)].copy()

    use_type_lock = len(type_pool) > 0
    category_pool = df_seed[df_seed['category'] == canonical_category].copy()
    working_pool  = type_pool if use_type_lock else category_pool
    lock_label    = f"food_type={scanned_food_type}" if use_type_lock else "category_only"

    valid_pool = working_pool[working_pool['thrs_v2_score'] >= 75].copy()
    confidence = "high-confidence"
    if len(valid_pool) < 1:
        valid_pool = working_pool[working_pool['thrs_v2_score'] >= 65].copy()
        confidence = "moderate-confidence (relaxed health floor)"
    if valid_pool.empty:
        valid_pool = category_pool[category_pool['thrs_v2_score'] >= 65].copy()
        confidence = "moderate-confidence (category fallback)"
        lock_label = "category_fallback"

    if valid_pool.empty:
        return {"scanned_product": item_name, "scanned_brand": scanned_row.get('brand'),
                "scanned_thrs": int(scanned_row.get('thrs_v2_score',0)),
                "category": canonical_category, "food_type": scanned_food_type,
                "guardrail_status": "no_healthy_alternatives_found", "recommendations": []}

    scanned_text = item_name + ' ' + str(scanned_row.get('key_difference',''))
    valid_pool = valid_pool.copy()
    valid_pool['match_text'] = (valid_pool['item_name'] + ' ' +
                                 valid_pool['ingredients_raw'].fillna('') + ' ' +
                                 valid_pool['key_difference'].fillna(''))

    all_texts  = [scanned_text] + valid_pool['match_text'].tolist()
    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
    tfidf_mat  = vectorizer.fit_transform(all_texts)
    tfidf_sims = cosine_similarity(tfidf_mat[0:1], tfidf_mat[1:]).flatten()
    tfidf_max  = tfidf_sims.max()
    norm_tfidf = tfidf_sims / tfidf_max if tfidf_max > 0 else tfidf_sims

    thrs_vals = valid_pool['thrs_v2_score'].values
    thrs_min, thrs_max = thrs_vals.min(), thrs_vals.max()
    norm_thrs = ((thrs_vals - thrs_min) / (thrs_max - thrs_min)
                 if thrs_max > thrs_min else np.ones(len(thrs_vals)))

    valid_pool['combined_score'] = 0.70 * norm_thrs + 0.30 * norm_tfidf
    valid_pool['tfidf_similarity'] = tfidf_sims
    top3 = valid_pool.sort_values('combined_score', ascending=False).head(3)

    recs = []
    for _, r in top3.iterrows():
        recs.append({"brand": r['brand'], "item_name": r['item_name'],
                     "thrs_v2_score": int(r['thrs_v2_score']),
                     "key_difference": r['key_difference'],
                     "tfidf_similarity": round(float(r['tfidf_similarity']),4),
                     "combined_score":   round(float(r['combined_score']),4)})

    return {"scanned_product": item_name, "scanned_brand": scanned_row.get('brand'),
            "scanned_thrs": int(scanned_row.get('thrs_v2_score',0)),
            "category": canonical_category, "food_type": scanned_food_type,
            "lock_applied": lock_label,
            "guardrail_status": confidence, "recommendations": recs}

print("Recommendation engine v3.1 ready.")


Recommendation engine v3.1 ready.


In [5]:
# Verify Yoga Bar Multigrain Energy Bar score
ybar = df_seed[df_seed['item_name'].str.contains('Multigrain Energy Bar', case=False)].iloc[0]
print("==================================================")
print("AUDIT: Yoga Bar Multigrain Energy Bar Score")
print("==================================================")
print(f"Item Name: {ybar['item_name']}")
print(f"Category : {ybar['category']}")
print(f"THRS Score: {ybar['thrs_v2_score']} / 100")


AUDIT: Yoga Bar Multigrain Energy Bar Score
Item Name: Yoga Bar Multigrain Energy Bar
Category : Confectionery / Sweets
THRS Score: 85 / 100


In [6]:
all_recs = []
for _, row in df_scanned.iterrows():
    all_recs.append(recommend_alternatives(row, df_seed))

with open('data/recommendations.json', 'w', encoding='utf-8') as f:
    json.dump(all_recs, f, indent=2, ensure_ascii=False)

print("Saved updated recommendations to data/recommendations.json")


Saved updated recommendations to data/recommendations.json
